# Positional Encoding

The transformer reads all words at once. It has no idea which
word comes first and which comes last. But word order matters.
The dog bit the man is very different from the man bit the dog.

So we need to stamp each word with its position before feeding
it to the model. The modern way to do this is called Rotary
Position Embeddings or RoPE for short. Instead of adding
position numbers RoPE rotates the vectors by an angle that
depends on where the word sits in the sentence.

The clever part is that after rotation the dot product between
any two words depends only on how far apart they are. Not on
their absolute positions. Words five slots apart always get the
same relative angle no matter if they appear at position 0 or
position 500. This is exactly what attention should care about.

This notebook shows you the rotation math and lets you watch
how a vector changes as it moves through different positions.

## Imports

In [ ]:
import torch
import torch.nn as nn
import math

## RoPE Implementation

RoPE expects input tensors in the shape [batch, num_heads, seq_len, head_dim].
This is the same layout used inside multi-head attention.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"

        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)

        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Watch rotation on a single vector

In [ ]:
rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=16)

q = torch.tensor([[[[0.8, 0.3, -0.5, 0.2]]]])  # [1, 1, 1, 4]
print(f"Original vector: {q[0, 0, 0].tolist()}")
print()

for pos in [0, 1, 2, 5]:
    rotated = rope(q, seq_len=pos + 1)
    last = rotated[0, 0, -1].tolist()
    vals = [f"{v:.3f}" for v in last]
    print(f"Position {pos}: [{', '.join(vals)}]")

## Relative position property

The dot product between two rotated vectors depends only on
their distance apart. Not on their absolute positions.

In [ ]:
rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

seq_len = 12
num_heads = 4
head_dim = 64

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len=seq_len)
k_rot = rope(k, seq_len=seq_len)

head = 0
a = (q_rot[0, head, 2] @ k_rot[0, head, 4]).item()
b = (q_rot[0, head, 5] @ k_rot[0, head, 7]).item()
c = (q_rot[0, head, 0] @ k_rot[0, head, 8]).item()

print(f"Distance 2 (positions 2,4):     {a:.4f}")
print(f"Distance 2 (positions 5,7):     {b:.4f}")
print(f"Distance 8 (positions 0,8):     {c:.4f}")
print()
print("Same distance gives similar score regardless of position.")

## RoPE inside attention

RoPE is applied only to queries and keys. Values stay untouched.
This makes the attention score position aware while keeping the
content clean.

In [ ]:
head_dim = 64
num_heads = 4
seq_len = 10

rope = RotaryPositionalEmbedding(d_model=head_dim, max_seq_len=128)

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len)
k_rot = rope(k, seq_len)

scores = (q_rot @ k_rot.transpose(-2, -1)) / math.sqrt(head_dim)
print(f"Attention scores shape: {scores.shape}")
print(f"Token 0 attends to itself:  {scores[0, 0, 0, 0]:.4f}")
print(f"Token 0 attends to token 5: {scores[0, 0, 0, 5]:.4f}")